In [ ]:
# 구글 스칼라 논문 수집하기

In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def extract_citation_count(text):
    """
    '590회 인용'과 같은 문자열에서 숫자 부분만 추출하는 함수
    """
    return int(text.split("회")[0].strip().replace(",", "")) if text.split("회")[0].strip().replace(",", "").isdigit() else 0

# Chrome WebDriver 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

driver.maximize_window()  # 창 최대화하여 숨겨진 요소 접근 가능하도록 설정

# 검색어 설정
query = "LLM"
base_url = "https://scholar.google.co.kr/scholar?hl=ko&as_sdt=0%2C5&q="

# 데이터 저장 리스트
titles = []  # 논문 제목
authors = []  # 저자 및 출판 정보
citations = []  # 인용 횟수
links = []  # 논문 링크

# 5페이지 크롤링 (한 페이지당 10개 논문)
for page_no in range(0, 5):
    page_url = f"{base_url}{query}&start={page_no * 10}"
    driver.get(page_url)
    time.sleep(3)  # 페이지 로딩 대기
    
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "gs_r"))
        )
        
        articles = driver.find_elements(By.CLASS_NAME, "gs_r")
        
        for article in articles:
            try:
                # 논문 제목 가져오기 (예외처리 추가)
                try:
                    title_element = article.find_element(By.CLASS_NAME, "gs_rt")
                    title = title_element.text
                except:
                    title = ""
                
                # 논문 링크 가져오기
                try:
                    link_element = title_element.find_elements(By.TAG_NAME, "a")
                    link = link_element[0].get_attribute("href") if link_element else ""
                except:
                    link = ""
                
                # 저자 및 출판 정보 가져오기
                try:
                    author_element = article.find_element(By.CLASS_NAME, "gs_a")
                    author = author_element.text
                except:
                    author = ""
                
                # 인용 횟수 가져오기 (개선된 방식 적용)
                try:
                    citation_element = article.find_element(By.XPATH, ".//a[contains(@href, '/scholar?cites=')]")
                    citation_count = extract_citation_count(citation_element.text)
                except:
                    citation_count = 0
                
                # 제목 또는 저자 정보가 없는 데이터는 저장하지 않음
                if title and author:
                    titles.append(title)
                    authors.append(author)
                    citations.append(citation_count)
                    links.append(link)
            
            except Exception as e:
                print(f"❌ 개별 논문 처리 중 오류 발생: {e}")
                continue
    
    except Exception as e:
        print(f"❌ 논문 리스트를 가져오는 중 오류 발생: {e}")

# 브라우저 종료
driver.quit()

# 데이터프레임 생성
df = pd.DataFrame({
    "번호": range(1, len(titles) + 1),  # 1부터 시작하는 번호 추가
    "논문 제목": titles,
    "저자 및 출판 정보": authors,
    "인용 횟수": citations,
    "논문 링크": links
})

# 데이터 확인
df.to_excel("google_scholar_scraped.xlsx", index=False)
print("크롤링 완료! 결과가 'google_scholar_scraped.xlsx' 파일로 저장되었습니다.")


크롤링 완료! 결과가 'google_scholar_scraped.xlsx' 파일로 저장되었습니다.
